# Training pipeline

## defining project path and the experiment name

In [ ]:
project_name = "building-footprint-segmentation"

In [ ]:
experiment_name = "ResUNet_resnet50"

## set path and load modules

In [ ]:
import os
import sys

module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import albumentations as A
from examples.environment import local_path as root
from building_footprint_segmentation.segmentation import init_segmentation, read_trainer_config
from building_footprint_segmentation.helpers.callbacks import CallbackList, load_callback

## load train configuration and initialize segmenter

In [ ]:
config_path = f"{root}/{project_name}/{experiment_name}.yaml"
config = read_trainer_config(config_path)

In [ ]:
segmentation = init_segmentation(config["Segmentation"])

## load objects for training

In [ ]:
model = segmentation.load_model(
    name=config["Model"]["name"],
    **config["Model"].get("param", {}),
)

In [ ]:
augmenters_alfa = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=1.0)
])


In [ ]:
loader = segmentation.load_loader(
    config["Loader"]["root"],
    config["Loader"]["image_normalizer"],
    config["Loader"]["label_normalizer"],
    augmenters_alfa,
    config["Loader"]["batch_size"],
)

In [ ]:
metrics = segmentation.load_metrics(
    data_metrics=config["Metrics"]
)

In [ ]:
optimizer = segmentation.load_optimizer(model, 
                                        name=config["Optimizer"]["name"], 
                                        lr= float(config["Optimizer"]["lr"]), 
                                        weight_decay = 1e-2)

In [ ]:
criterion = segmentation.load_criterion(**config["Criterion"])

In [ ]:
callbacks = CallbackList(base_log_dir = config["Callbacks"]["log_dir"])
# Ouptut from all the callbacks caller will be stored at the path specified in log_dir
for caller in config["Callbacks"]["callers"]:
    callbacks.append(load_callback(None, caller))

## initialize trainer

In [ ]:
# 1. Import the new visualization classes
from building_footprint_segmentation.trainer import EnhancedTrainingVisualizer
from building_footprint_segmentation.helpers.callbacks import VisualizationCallback, AUCCallback
from building_footprint_segmentation.trainer import Trainer
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 2. Initialize the visualizer
visualizer = EnhancedTrainingVisualizer(save_dir=callbacks.base_log_dir)

# 3. Create the callbacks
viz_callback = VisualizationCallback(
    log_dir=callbacks.log_dir,
    visualizer=visualizer,
    plot_frequency=5,  # Plot every 5 epochs
    collect_auc_data=True
)

auc_callback = AUCCallback(
    log_dir=callbacks.log_dir,
    visualizer=visualizer,
    compute_frequency=10  # Compute AUC every 10 epochs
)

# 4. Add these callbacks to your existing callbacks
callbacks.append(viz_callback)
callbacks.append(auc_callback)

# 5. set up a strategy to reduce the learning rate
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    threshold=1e-4,
    threshold_mode='rel',
    cooldown=0,
    min_lr=1e-6,
)

# 6. Now create the trainer with all callbacks (including the new ones)
trainer = Trainer(
    model=model,
    criterion=criterion,
    loader=loader,
    metrics=metrics,
    callbacks=callbacks,  # This now includes visualization callbacks
    optimizer=optimizer,
    scheduler=scheduler,
)

# 7. IMPORTANT: Update callback parameters to pass the loader
callbacks.update_params({'loader': loader})

## train a new model

In [ ]:
trainer.train(start_epoch=0, end_epoch=100)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

exps = sorted(
    [p for p in Path(f"{root}/{project_name}/logs/{experiment_name}").iterdir() if p.is_dir()], 
    reverse=True
)

for exp in exps:
    print(f"Experimento iniciado em {exp.name}:")
    display(Image(filename=exp/'final_training_curves.png')) 

# Evaluate models

In [ ]:
from building_footprint_segmentation.evaluator import Evaluator, PlotSelection
from building_footprint_segmentation.seg.binary.models import ResUNetResNet50

loader = segmentation.load_loader(
    config["Loader"]["root"],
    config["Loader"]["image_normalizer"],
    config["Loader"]["label_normalizer"],
    augmenters_alfa,
    1,
)

#sel = PlotSelection(n=5, mode="alpha")
sel = PlotSelection(n=5, mode="random", seed=42, seed_per_experiment=False)
#sel = PlotSelection(n=5, mode="random", seed=0, seed_per_experiment=True)

evaluator = Evaluator(
    base_dir=f"{root}/{project_name}/logs/{experiment_name}",
    model_factory=lambda: ResUNetResNet50(),
    test_loader=loader.test_loader,
    metrics=metrics,
    threshold=0.45,
)

report = evaluator.run(
    selection=sel,
    save_plots=True,            # save in <exp>/test/<stem>_eval.png
    show_plots=True,            # True on Jupyter
    output_json=f"{root}/{project_name}/logs/{experiment_name}/evaluation_summary.json",
)